In [30]:
# Importação dos módulos
import pandas as pd
import requests
import os
from dotenv import load_dotenv

In [31]:
# Leitura de variáveis e parâmetros

load_dotenv()

BASE_URL = "https://app.kickoffapi.com/api/v1"
API_KEY = os.getenv('KICK_OFF_API_KEY')

HEADERS = {
    'x-api-key': API_KEY
}

PARAMS = {
    'league': 1,
    'season': 2026
}

ENDPOINTS = {
    "fixtures": f"{BASE_URL}/fixtures",
    "teams": f"{BASE_URL}/teams",
    "players": f"{BASE_URL}/players",
    "teams_statistics": f"{BASE_URL}/fixtures/{id}/statistics",
    "events": f"{BASE_URL}/fixtures{id}/events",
    "league": f"{BASE_URL}/leagues"
}


In [32]:
# Função para requisição HTTP

def get_data(endpoint):

    resposta = requests.get(
        endpoint,
        headers=HEADERS,
        params=PARAMS,
        timeout=30
    )

    resposta.raise_for_status()

    return resposta.json()

In [33]:
# Criação do Dataframe dos fatos
df_fixtures = pd.json_normalize(get_data(f"{BASE_URL}/fixtures")["response"])
fixtures = df_fixtures[
    [
        "id",
        "date",
        "leagueId",
        "venueId",
        "homeTeamId",
        "awayTeamId",
        "goalsHome",
        "goalsAway",
        "scoreHalfHome",
        "scoreHalfAway",
        "scoreFullHome",
        "scoreFullAway",
        "seasonYear",
        "statusLong"
    ]
]

In [34]:
# Criação do Dataframe da dimensão dos times/seleções
home = df_fixtures[
    [
        "homeTeam.id",
        "homeTeam.name",
        "homeTeam.logo"
    ]
]

home.columns = [
    "id",
    "name",
    "logo"
]

away = df_fixtures[
    [
        "awayTeam.id",
        "awayTeam.name",
        "awayTeam.logo"
    ]
]

away.columns = [
    "id",
    "name",
    "logo"
]

teams = pd.concat([home, away]).drop_duplicates()

In [35]:
# Criação do Dataframe da dimensão dos estádios
venues = df_fixtures[
    [
        "venue.id",
        "venue.name",
        "venue.city",
    ]
]

venues.columns = [
    "id",
    "name",
    "city"
]

venues = venues.drop_duplicates()

In [36]:
# Criação do Dataframe da dimensão dos jogadores
df_players = pd.json_normalize(get_data(f"{BASE_URL}/players")["response"])

players = df_players[
    [
        "id",
        "name",
        "age",
        "nationality",
        "height",
        "weight"
    ]
]

players = players.drop_duplicates()

In [ ]:
# Criação do Dataframe da dimensão das estatísticas dos jogos
data = []

for fixture in fixtures["id"][:2]:
  response = get_data(f"{BASE_URL}/fixtures/{fixture}/statistics")["response"]
  data.extend(response)

df_fixtures_stats = pd.json_normalize(data)

fixtures_stats = df_fixtures_stats[
    [
        "id",
        "fixtureId",
        "teamId",
        "type",
        "value"
    ]
]

fixtures_stats = fixtures_stats.drop_duplicates()

In [ ]:
# Criação do Dataframe da dimensão dos eventos dos jogos
data = []

for fixture in fixtures["id"][:2]:
  response = get_data(f"{BASE_URL}/fixtures/{fixture}/events")["response"]
  data.extend(response)

df_events = pd.json_normalize(data)

events = df_events[
    [
        "id",
        "fixtureId",
        "time",
        "playerId",
        "assistId",
        "teamId",
        "type",
        "detail",
        "comments"
    ]
]

events = events.drop_duplicates()